# Deduplicate ICAEW full list against accountants-template

In [8]:
import pandas as pd
import os

ICAEW_PATH     = os.path.expanduser("~/Downloads/icaew_chartered_accountants_UK_full.csv")
TEMPLATE_PATH  = os.path.join(os.path.dirname(os.getcwd()), "leadmagnet-taxready", "accountants-template.csv")

# Adjust TEMPLATE_PATH if running from within the repo directory
if not os.path.exists(TEMPLATE_PATH):
    TEMPLATE_PATH = os.path.join(os.getcwd(), "accountants-template.csv")

icaew    = pd.read_csv(ICAEW_PATH, encoding="utf-8-sig")
template = pd.read_csv(TEMPLATE_PATH, encoding="utf-8-sig")

print(f"ICAEW rows:    {len(icaew)}")
print(f"Template rows: {len(template)}")
print(f"\nICAEW columns:    {list(icaew.columns)}")
print(f"Template columns: {list(template.columns)}")

ICAEW rows:    7895
Template rows: 2690

ICAEW columns:    ['Name', 'Type', 'Specialisms', 'Address', 'Distance']
Template columns: ['place_id', 'name', 'address', 'suburb', 'city', 'rating', 'reviews', 'longitude', 'latitude', 'postcode', 'outward_code', 'xero_hospitality', 'xero_construction', 'xero_healthcare', 'xero_media', 'xero_professional_services', 'xero_real_estate', 'xero_retail']


In [9]:
# Normalise firm names for comparison: lowercase + strip whitespace
def normalise(s):
    return str(s).strip().lower()

existing_names = set(template["name"].apply(normalise))

icaew["_name_norm"] = icaew["Name"].apply(normalise)

is_duplicate = icaew["_name_norm"].isin(existing_names)

duplicates   = icaew[is_duplicate]
deduplicated = icaew[~is_duplicate].drop(columns=["_name_norm"])

print(f"Duplicates found:  {len(duplicates)}")
print(f"Remaining rows:    {len(deduplicated)}")

if len(duplicates) > 0:
    print("\nDuplicate firm names:")
    print(duplicates["Name"].tolist())

Duplicates found:  50
Remaining rows:    7845

Duplicate firm names:
['Alliotts LLP', 'Hillier Hopkins LLP', 'Percy Westhead & Company', 'Accountax Partners Ltd', 'Advantage Accountancy & Advisory Ltd', 'Agnitio Accountants Ltd', 'Keith Willis Associates Limited', 'Page Kirk LLP', 'Bexons Accountants Limited', 'Hurren & Jubb Accountants Limited', 'Robson Laidler Accountants Limited', 'Troy Accounting Ltd', 'Vibrant Accountancy Limited', 'Marshall Accountancy Ltd', 'Johnston Wood Roach Limited', 'Candour Accounts Limited', 'RIGHT RATIOS LTD', 'CED Accountancy Services Limited', 'KRW Accountants Ltd', 'Merlin Accountancy Services Ltd', 'Sidaways Limited', 'Simpkins Edwards LLP', 'Watermill Accounting Limited', 'Randall & Payne LLP', 'Surrey Hills Accountancy Limited', 'TaxAssist Accountants', 'Everyday Accountants', 'Fair View Accounting & Tax Services Ltd', 'ANEES BELLA ACCOUNTANTS LTD', 'Radford & Sergeant Limited', 'In Accountancy Limited', 'Saddique & Co', 'Charlton Baker Limited', '

In [10]:
OUTPUT_PATH = os.path.expanduser("~/Downloads/icaew_chartered_accountants_UK_deduped.csv")
deduplicated.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"Saved {len(deduplicated)} rows to: {OUTPUT_PATH}")

Saved 7845 rows to: C:\Users\lhkan1/Downloads/icaew_chartered_accountants_UK_deduped.csv
